# Mortgage Doc RAG — Colab Demo

End-to-end demo: dual-engine OCR → document separation → validated RAG → layered evals.

Runtime: any GPU works (T4 is fine; L4/A100 pick a larger model automatically). CPU works too, slowly.

In [9]:
# Install (Colab)
!pip install -q uv
!git clone -q https://github.com/tccao/mortgage-doc-rag.git /content/mortgage-doc-rag 2>/dev/null || git -C /content/mortgage-doc-rag pull -q
%cd /content/mortgage-doc-rag
# llama-cpp-python needs a current version (older releases can't load newer GGUF
# architectures) AND CUDA support. Prebuilt CUDA wheels are stale (0.2.x), so
# source-build with CUDA (~15-25 min) unless a good build is already present.
!python -c "import llama_cpp; assert llama_cpp.llama_supports_gpu_offload(); assert tuple(map(int, llama_cpp.__version__.split('.')[:2])) >= (0, 3)" 2>/dev/null \
 || CMAKE_ARGS="-DGGML_CUDA=on" uv pip install --system -q --reinstall --no-deps --no-cache llama-cpp-python
# llama_cpp.server runtime deps, installed directly so the resolver never
# touches (and CPU-rebuilds) the CUDA llama-cpp-python build.
!uv pip install --system -q uvicorn fastapi sse-starlette starlette-context pydantic-settings
# Non-editable on purpose: editable installs need a kernel restart to be importable.
# --reinstall-package so a pulled update always replaces the installed copy.
!uv pip install --system -q --reinstall-package mortgage-rag ".[llm,ui]"
# The resolver may upgrade torch past what Colab's preinstalled torchvision/torchaudio
# were built for ("operator torchvision::nms does not exist"). Neither is used here.
!uv pip uninstall --system -q torchvision torchaudio
!python -c "import llama_cpp; print('llama-cpp-python', llama_cpp.__version__, '| GPU offload:', llama_cpp.llama_supports_gpu_offload())"

/content/mortgage-doc-rag
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 40441 MiB):
  Device 0: NVIDIA A100-SXM4-40GB, compute capability 8.0, VMM: yes, VRAM: 40441 MiB
llama-cpp-python 0.3.34 | GPU offload: True


In [10]:
# Pick model tier from detected GPU
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    gpu, vram_gb = "cpu", 0

if vram_gb >= 20:  # L4 / A100 tier
    repo, filename = "deepreinforce-ai/Ornith-1.0-35B-GGUF", "ornith-1.0-35b-Q4_K_M.gguf"
else:              # T4 tier / CPU
    repo, filename = "TheBloke/Mistral-7B-Instruct-v0.1-GGUF", "mistral-7b-instruct-v0.1.Q4_K_M.gguf"

print(f"{gpu} ({vram_gb:.0f} GB) -> {repo}")

NVIDIA A100-SXM4-40GB (42 GB) -> deepreinforce-ai/Ornith-1.0-35B-GGUF


In [11]:
# Process the adversarial loan-file bundle
from mortgage_rag import PipelineConfig, process_files, build_orchestrator

cfg = PipelineConfig.load(
    llm_backend="openai_compat",
    llm_api_base="http://127.0.0.1:8000/v1",
    llm_model_name=filename,
)
result = process_files(["data/loan_files/loan_file_01_conflicting_cd.pdf"], cfg)

print(result.stats)
print(result.consistency_report)
for f in result.files:
    for issue in f.validation_issues:
        print("!", issue)

{'files_processed': 1, 'total_pages': 23, 'total_documents': 11, 'total_chunks': 23, 'doc_type_breakdown': {'Mortgage Contract': 5, 'Loan Estimate': 1, 'Closing Disclosure': 1, 'Tax Document': 2, 'Pay Slip': 2}, 'errors': 0}
Loan amounts consistent: $285,803.36


In [12]:
# Serve the model once; the demo cells and the benchmark subprocess share it over
# HTTP. Two in-process copies of the 35B tier would exceed 40 GB VRAM.
import subprocess
import time
import urllib.request

from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id=repo, filename=filename)
server = subprocess.Popen(
    ["python", "-m", "llama_cpp.server", "--model", model_path,
     "--n_gpu_layers", "-1", "--n_ctx", "4096", "--port", "8000"],
    stdout=open("/tmp/llama_server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2)
        print("server up")
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("server exited; see /tmp/llama_server.log")
        time.sleep(5)
else:
    raise RuntimeError("server did not come up; see /tmp/llama_server.log")

server up


In [13]:
# Ask questions (note the conflicting 'corrected' closing disclosure in this bundle)
orch = build_orchestrator(result.index, cfg)

for q in ["What is the loan amount on the closing disclosure?",
          "What is the interest rate?"]:
    res = orch.answer(q)
    print(f"Q: {q}\nA: {res.answer}")
    for c in res.citations[:2]:
        print(f"   source: {c.doc_type} p{c.page_start} (score {c.score:.2f})")
    print()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Q: What is the loan amount on the closing disclosure?
A: Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** What is the loan amount on the closing disclosure?
   - **Constraint:** Answer briefly using only the context below. If the context does not contain the answer, say so.
   - **Context Provided:** Three sections from mortgage documents:
     - [Mortgage Contract, pages 11-11]: Closing Disclosure CORRECTED COPY -> Loan Amount: $150,000.00
     - [Closing Disclosure, pages 7-8]: Shows "Loan Amount $162,000.00" in section 02. Also mentions other figures like Sale Price $180,000, Down Payment $18,000, etc.
     - [Mortgage Contract, pages 5-5]: Sample Closing Disclosure description (no specific loan amount listed).

2.  **Identify Relevant Information:**
   - The context contains two different closing disclosures with different loan amounts:
     - Corrected copy: $150,000.00
     - Standard copy (pages 7-8): $162,000.00
   - I need to report what's in the cont

In [14]:
# Run the eval harness (retrieval layer is deterministic and fast)
!python evals/run_eval.py --retrieval-only
!head -40 evals/report.md

== Retrieval layer ==
Loading weights: 100% 199/199 [00:00<00:00, 6415.13it/s]
LLM is explicitly disabled. Using MockLLM.
   retrieval: hit@k 100.0% | MRR 0.862
   [45.0s]

Report written to /content/mortgage-doc-rag/evals/report.md
# Evaluation report

- Run: 2026-07-20 00:19:45 | mode: classical | backend: none | model: none | device: NVIDIA A100-SXM4-40GB | commit: fde7c36
- Corpus: 64 clean + 64 degraded PDFs, 3 adversarial bundles, 13 doc types
- Layer runtime: rag 45.0s

| Layer | Metric | Value |
|---|---|---|
| Retrieval | hit@k / MRR over 29 cases | 100.0% / 0.862 |


## Full benchmark

Run this on an L4/A100. It writes `evals/report.md` and `evals/baselines/latest.json`, which are
what the README results table and `docs/design.md` cite.

> The saved output below is the 2026-07-20 run, made **with** the reranker enabled. Re-running
> now measures the current config (reranker off), so expect the answer pass rate to move —
> most likely up, since two failures were caused by reranking hiding their gold documents.
> That is a prediction, not a measurement: retrieval reaching the model is necessary but not
> sufficient for those cases to pass.

In [15]:
# Full benchmark (long — L4/A100). Writes evals/report.md + evals/baselines/latest.json.
# Talks to the shared model server started above; --resume reuses layers a crashed run finished.
import os
import urllib.request

os.environ["MRAG_LLM_BACKEND"] = "openai_compat"
os.environ["MRAG_LLM_API_BASE"] = "http://127.0.0.1:8000/v1"
os.environ["MRAG_LLM_MODEL_NAME"] = filename

# Fail fast: the OCR layer alone is ~25 min, so catch a dead server before paying for it.
urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5)

# Print the config the report will be attributed to. The reranker line matters — the
# committed report predates ADR-10 and was produced with it enabled.
from mortgage_rag.config import PipelineConfig

cfg = PipelineConfig.load()
print(f"model={cfg.llm_model_name}  backend={cfg.llm_backend}  mode={cfg.mode}")
print(f"use_reranker={cfg.use_reranker}  top_k={cfg.top_k}  rerank_top_n={cfg.rerank_top_n}")
print(f"chunk_size={cfg.chunk_size}  overlap={cfg.chunk_overlap}  temperature={cfg.temperature}")

!python -u evals/run_eval.py --all --save-baseline --resume
!cat evals/report.md

== OCR layer (degraded scans vs ground truth) ==
   mean CER 0.7033 | mean WER 0.9308 over 64 files [1469.5s]
== Classification layer ==
   clean: 94.6% (53/56)
   degraded: 92.9% (52/56)
   [0.0s]
== Retrieval + answer layer ==
Loading weights: 100% 199/199 [00:00<00:00, 5534.63it/s]
LLM is explicitly disabled. Using MockLLM.
Loading weights: 100% 105/105 [00:00<00:00, 6106.35it/s]
Loading weights: 100% 105/105 [00:00<00:00, 4942.45it/s]
Loading weights: 100% 105/105 [00:00<00:00, 4399.22it/s]
Loading weights: 100% 105/105 [00:00<00:00, 6104.57it/s]
Loading weights: 100% 105/105 [00:00<00:00, 6015.27it/s]
   retrieval: hit@k 100.0% | MRR 0.862
   answer: pass 69.2% | adversarial resistance 0.8
   [254.9s]

Report written to /content/mortgage-doc-rag/evals/report.md
Baseline saved to /content/mortgage-doc-rag/evals/baselines/latest.json
# Evaluation report

- Run: 2026-07-20 00:20:34 | mode: classical | backend: openai_compat | model: ornith-1.0-35b-Q4_K_M.gguf | device: NVIDIA A100-SX